# 04 — GeoPandas dan morfologi kota

GeoPandas adalah pandas dengan kolom geometri. Di notebook ini: baca data, disiplin CRS, lalu metrik bentuk
kota per kelurahan — jumlah dan kepadatan bangunan, rasio tutupan, ukuran blok, dan kekompakan.

In [ ]:
import sys
sys.path.insert(0, "../scripts")

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
from shapely.ops import polygonize, unary_union

from kota import CFG, OSM, PROC, RAW, UTM

kota = gpd.read_file(OSM, layer="batas_kota")
kel = gpd.read_file(OSM, layer="kelurahan")
bangunan = gpd.read_file(OSM, layer="bangunan")
print(len(kel), "kelurahan |", len(bangunan), "bangunan |", kel.crs)

## 1. Disiplin CRS
Data OSM disimpan dalam derajat (EPSG:4326). Luas dalam "derajat persegi" tidak bermakna — dan tidak ada
peringatan yang menghentikan Anda menghitungnya. Proyeksikan dulu ke UTM dari konfigurasi.

In [ ]:
print("Luas di EPSG:4326 :", round(kota.area.iloc[0], 5), "(derajat persegi — salah)")
kota_m = kota.to_crs(UTM)
print(f"Luas di {UTM}:", round(kota_m.area.iloc[0] / 1e6, 1), "km²")
print(f"Luas resmi ({CFG['luas_resmi_sumber']}): {CFG['luas_resmi_km2']} km² — batas OSM tidak identik dengan batas resmi")

In [ ]:
kel = kel.to_crs(UTM)
bangunan = bangunan.to_crs(UTM)
kel["luas_ha"] = kel.area / 1e4
bangunan["tapak_m2"] = bangunan.area
bangunan["tapak_m2"].describe().round(1)

## 2. Bangunan per kelurahan
*Spatial join* dengan titik representatif bangunan, agar bangunan di garis batas masuk ke satu kelurahan saja.

In [ ]:
titik = bangunan[["tapak_m2"]].set_geometry(bangunan.representative_point())
j = gpd.sjoin(titik, kel[["id", "geometry"]], predicate="within", how="inner")
agg = j.groupby("id").agg(n_bangunan=("tapak_m2", "size"), tapak_total_m2=("tapak_m2", "sum"),
                            tapak_median_m2=("tapak_m2", "median"))
kel = kel.merge(agg, left_on="id", right_index=True, how="left").fillna({"n_bangunan": 0, "tapak_total_m2": 0})
kel["bangunan_per_ha"] = kel["n_bangunan"] / kel["luas_ha"]
kel["rasio_tutupan"] = kel["tapak_total_m2"] / (kel["luas_ha"] * 1e4)
print("Bangunan di luar semua kelurahan:", len(titik) - len(j))
kel[["kelurahan", "kecamatan", "luas_ha", "n_bangunan", "bangunan_per_ha", "rasio_tutupan", "tapak_median_m2"]] \
    .sort_values("bangunan_per_ha", ascending=False).head(10).round(2)

## 3. Blok kota dari jaringan jalan
Blok = poligon yang dikelilingi jalan. Kita bentuk dengan `polygonize` pada ruas jalan (jaringan `drive`
ditambah jalan lingkungan), lalu buang poligon yang terlalu kecil (sisa persimpangan) atau terlalu besar.

In [ ]:
G = ox.load_graphml(RAW / "jalan_walk.graphml")
edges = ox.graph_to_gdfs(G, nodes=False)[["highway", "geometry"]].to_crs(UTM)
garis = unary_union(edges.geometry.values)
blok = gpd.GeoDataFrame(geometry=list(polygonize(garis)), crs=UTM)
blok = gpd.clip(blok, kota_m)
blok["luas_m2"] = blok.area
blok = blok[(blok["luas_m2"] >= 500) & (blok["luas_m2"] <= 500_000)].reset_index(drop=True)
blok["kekompakan"] = 4 * np.pi * blok.area / blok.length ** 2
print(len(blok), "blok | median luas", round(blok["luas_m2"].median()), "m²")

In [ ]:
bp = blok.set_geometry(blok.representative_point())
bp = gpd.sjoin(bp, kel[["id", "geometry"]], predicate="within")
kel = kel.merge(bp.groupby("id").agg(blok_median_m2=("luas_m2", "median"),
                                       kekompakan_blok=("kekompakan", "median")),
                left_on="id", right_index=True, how="left")
blok = blok.join(bp["id"].rename("kel_id"))
blok.to_file(PROC / "blok.gpkg", layer="blok")

## 4. Membandingkan dua kelurahan
Angka kepadatan tidak menafsirkan dirinya sendiri. Bandingkan kelurahan terpadat dengan yang paling jarang.
Nama kelurahan tidak unik (Semarang punya dua Purwosari), jadi label selalu memuat kecamatannya.

In [ ]:
kel["label"] = kel["kelurahan"] + " (" + kel["kecamatan"] + ")"
cols = ["luas_ha", "n_bangunan", "bangunan_per_ha", "rasio_tutupan", "tapak_median_m2", "blok_median_m2",
        "kekompakan_blok"]
urut = kel.sort_values("bangunan_per_ha", ascending=False)
urut.iloc[[0, -1]].set_index("label")[cols].T.round(2)

In [ ]:
ax = kel.plot(column="bangunan_per_ha", cmap="magma_r", legend=True, figsize=(9, 7),
              legend_kwds={"label": "bangunan per ha"})
ax.set_axis_off()
ax.set_title(f"Kepadatan bangunan per kelurahan — {CFG['name']}")

In [ ]:
kel.to_file(PROC / "morfologi.gpkg", layer="kelurahan")
bangunan.to_file(PROC / "bangunan_utm.gpkg", layer="bangunan")
print("disimpan:", PROC.name, "/ morfologi.gpkg, blok.gpkg, bangunan_utm.gpkg")